In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import time

# Loading the 2015 Flight Delays dataset
flights = pd.read_csv('/kaggle/input/datasets/organizations/usdot/flight-delays/flights.csv', low_memory=False)
df_sample = flights.head(2000).copy()

# Creating the mapping and the Graph (mat)
all_airports = pd.concat([df_sample['ORIGIN_AIRPORT'], df_sample['DESTINATION_AIRPORT']]).unique()
airport_to_idx = {code: i + 1 for i, code in enumerate(all_airports)}
idx_to_airport = {i + 1: code for i, code in enumerate(all_airports)}

mat = []
for index, row in df_sample.iterrows():
    if row['ORIGIN_AIRPORT'] in airport_to_idx and row['DESTINATION_AIRPORT'] in airport_to_idx:
        mat.append([airport_to_idx[row['ORIGIN_AIRPORT']], 
                    airport_to_idx[row['DESTINATION_AIRPORT']], 
                    row['DISTANCE']])

n = len(all_airports)
print(f"Dataset ready: {n} airports mapped.")

In [ ]:
class MinHeapPQ():
    """Class to execute priority queue functions using a Min-Heap."""
    def __init__(self):
        self.heap_size = 0
        self.heap = []
        
    def get_parent(self, i): return int((i - 1) / 2)
    def get_left(self, i): return int(2 * i + 1)
    def get_right(self, i): return int(2 * i + 2)
    
    def min_heapify(self, i):
        l, r = self.get_left(i), self.get_right(i)
        smallest = l if l < self.heap_size and self.heap[l][1] < self.heap[i][1] else i
        if r < self.heap_size and self.heap[r][1] < self.heap[smallest][1]:
            smallest = r
        if smallest != i:
            self.heap[i], self.heap[smallest] = self.heap[smallest], self.heap[i]
            self.min_heapify(smallest)
            
    def insert(self, key_value):
        self.heap.append(key_value)
        self.heap_size = len(self.heap)
        self.decrease_key(self.heap_size - 1)

    def decrease_key(self, index):
        while (index > 0) and (self.heap[self.get_parent(index)][1] > self.heap[index][1]):
            p = self.get_parent(index)
            self.heap[p], self.heap[index] = self.heap[index], self.heap[p]
            index = p
        
    def update_weight(self, key_value):
        for i in range(len(self.heap)):
            if self.heap[i][0] == key_value[0]:
                self.heap[i][1] = key_value[1]
                self.decrease_key(i)
                break

    def extract_min(self):
        if self.heap_size == 0: return None
        root = self.heap[0]
        last = self.heap.pop()
        self.heap_size = len(self.heap)
        if self.heap_size > 0:
            self.heap[0] = last
            self.min_heapify(0)
        return root
            
    def is_empty(self): return self.heap_size == 0

In [98]:
import heapq
class Dijkstra:
    def __init__(self, n):
        self.n = n
        self.distances = []

    def run(self, adj_list, source):
        self.distances = [float('inf')] * (self.n + 1)
        self.distances[source] = 0
        pq = [(0, source)]

        while pq:
            current_dist, u = heapq.heappop(pq)

            if current_dist > self.distances[u]:
                continue

            # Check if the airport exists in our adjacency list
            if u in adj_list:
                for v, w in adj_list[u]:
                    if self.distances[u] + w < self.distances[v]:
                        self.distances[v] = self.distances[u] + w
                        heapq.heappush(pq, (self.distances[v], v))
        return self.distances
        
# Dijkstra needs to know which airports connect to which
adj = {}
for u, v, w in mat:
    if u not in adj: adj[u] = []
    if v not in adj: adj[v] = []
    # Adding both directions for undirected flight paths
    adj[u].append((v, w))
    adj[v].append((u, w))

n = len(all_airports)
source_node = mat[0][0]

dijkstra_solver = Dijkstra(n)
start_dij = time.time() * 1000
dij_results = dijkstra_solver.run(adj, source_node)
end_dij = time.time() * 1000

print(f"Algorithm: Dijkstra")
print(f"Execution Time: {end_dij - start_dij:.4f} ms")

for i in range(1, 6):
    airport_code = idx_to_airport.get(i, "Unknown")
    print(f"Airport {airport_code}: Distance {dij_results[i]}")

Algorithm: Dijkstra
Execution Time: 0.7761 ms
Airport ANC: Distance 0
Airport LAX: Distance 2376
Airport SFO: Distance 2092
Airport SEA: Distance 1448
Airport LAS: Distance 2305


In [94]:
class BellmanFord:
    def __init__(self, n):
        self.n = n
        self.distances = [float('inf')] * (n + 1)

    def run(self, edges, start_node):
        self.distances[start_node] = 0
        
        for i in range(self.n - 1):
            changed = False
            for u, v, w in edges:
                # Check Forward: U -> V
                if self.distances[u] != float('inf') and self.distances[u] + w < self.distances[v]:
                    self.distances[v] = self.distances[u] + w
                    changed = True
                # Check Backward: V -> U (important for undirected flights)
                if self.distances[v] != float('inf') and self.distances[v] + w < self.distances[u]:
                    self.distances[u] = self.distances[v] + w
                    changed = True
            
            # Checking if no distances changed in a full pass
            if not changed:
                print(f"Converged early at iteration {i}")
                break
        
        return self.distances
bf_solver = BellmanFord(n)
source_node = mat[0][0] 

start_bf = time.time() * 1000
bf_results = bf_solver.run(mat, source_node)
end_bf = time.time() * 1000

# Printing the first 5 results to check the distance
for i in range(1, 6):
    print(f"Airport {idx_to_airport[i]}: Distance {bf_results[i]}")

print(f"\nFinal Bellman-Ford Time: {end_bf - start_bf:.2f} ms")


Converged early at iteration 2
Airport ANC: Distance 0
Airport LAX: Distance 2376
Airport SFO: Distance 2092
Airport SEA: Distance 1448
Airport LAS: Distance 2305

Final Bellman-Ford Time: 3.44 ms
